In [4]:
import pandas as pd
import sqlite3
import json
import requests

# 1. Descargar el archivo desde el enlace compartido de Google Drive
file_id = "17Kl8hxFSykR9qcBA8JEMUU568Mv_8vuS"
archivo_excel = "DB_Estacion_Fija_Tomate_Quindio.xlsx"

print("Descargando el archivo desde Google Drive...")
download_url = f"https://docs.google.com/spreadsheets/d/{file_id}/export?format=xlsx"
r = requests.get(download_url)
with open(archivo_excel, 'wb') as f:
    f.write(r.content)
print("¡Descarga completada!")

# Leer el archivo Excel descargado
df = pd.read_excel(archivo_excel)

# 2. Eliminar columnas que no necesitamos
columnas_a_eliminar = ["RGB_capture", "Tapo_capture", "Parrot_capture"]
df = df.drop(columns=[col for col in columnas_a_eliminar if col in df.columns])

# 3. Función auxiliar para convertir NaN a None (manejo de datos en blanco)
def clean_val(val):
    if pd.isna(val):
        return None
    return val

# 4. Conectar a SQLite (creará el archivo si no existe)
db_path = "agrotech_migracion.db"
conn = sqlite3.connect(db_path)
cursor = conn.cursor()

# 5. Crear la tabla de mediciones (Agregamos radiacion REAL)
cursor.execute("""
    CREATE TABLE IF NOT EXISTS mediciones (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        nodo_id TEXT NOT NULL,
        timestamp TEXT NOT NULL,
        temperatura REAL,
        humedad REAL,
        nitrogeno REAL,
        fosforo REAL,
        potasio REAL,
        humedad_suelo REAL,
        temperatura_suelo REAL,
        conductividad REAL,
        ph REAL,
        bateria REAL,
        radiacion REAL,
        payload TEXT NOT NULL,
        enviado INTEGER NOT NULL DEFAULT 0
    )
""")

# 6. Insertar los datos iterando sobre las filas del Excel
for index, row in df.iterrows():
    timestamp = str(row['Time']) if not pd.isna(row['Time']) else "1970-01-01 00:00:00"

    # Extraer valores limpios
    t_amb = clean_val(row.get('Air_sensor_Temperature[°C]'))
    h_amb = clean_val(row.get('Air_sensor_Humidity[%RH]'))
    n = clean_val(row.get('7in1_Nitrogen[mg/kg]'))
    p = clean_val(row.get('7in1_Phosphorus[mg/kg]'))
    k = clean_val(row.get('7in1_Potasium[mg/kg]'))
    hum_s = clean_val(row.get('7in1_Moisture[%RH]'))
    temp_s = clean_val(row.get('7in1_S_Temperature[°C]'))
    ec = clean_val(row.get('7in1_Conductivity[uS/cm]'))
    ph = clean_val(row.get('7in1_Ph[pH]'))
    rad = clean_val(row.get('Pynamometer_Radiation[W/m^2]'))
    bateria = None # No hay batería en el Excel

    # Crear el JSON compatible con la aplicación Agrotech
    payload_dict = {
        "tipo": "lectura",
        "dht": {"t": t_amb, "h": h_amb},
        "suelo": {"N": n, "P": p, "K": k, "hum": hum_s, "temp": temp_s, "ec": ec, "ph": ph},
        "piramometro": {"rad": rad},
        "bateria": {"porcentaje": bateria}
    }

    payload_json = json.dumps(payload_dict, ensure_ascii=False)

    # Insertar a la BD (El campo 'enviado' queda en 0 por defecto de la tabla)
    cursor.execute("""
        INSERT INTO mediciones (
            nodo_id, timestamp, temperatura, humedad, nitrogeno, fosforo, potasio,
            humedad_suelo, temperatura_suelo, conductividad, ph, bateria, radiacion, payload
        ) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
    """, (
        "nodo1", timestamp, t_amb, h_amb, n, p, k,
        hum_s, temp_s, ec, ph, bateria, rad, payload_json
    ))

conn.commit()
conn.close()
print("¡Migración a base de datos SQLite completada con éxito!")

Descargando el archivo desde Google Drive...
¡Descarga completada!
¡Migración a base de datos SQLite completada con éxito!


In [5]:
import os

# Verificar la ubicación de los archivos clave en el entorno local de Colab
archivos = ["DB_Estacion_Fija_Tomate_Quindio.xlsx", "agrotech_migracion.db"]

for archivo in archivos:
    if os.path.exists(archivo):
        print(f"El archivo '{archivo}' se encuentra en: {os.path.abspath(archivo)}")
    else:
        print(f"El archivo '{archivo}' no se encontró en el directorio actual.")

El archivo 'DB_Estacion_Fija_Tomate_Quindio.xlsx' se encuentra en: /content/DB_Estacion_Fija_Tomate_Quindio.xlsx
El archivo 'agrotech_migracion.db' se encuentra en: /content/agrotech_migracion.db
